# Bank Loan Approval Data Exploration

## 1. Introduction
This project performs Exploratory Data Analysis (EDA) on bank loan application data to understand which factors are associated with loan approval or rejection.

**Objectives**
- Understand customer profiles in the loan application data
- Find patterns between approved and rejected applications
- Explore the relationship of income, credit history, loan amount and property area with approval
- Detect missing values, duplicates and outliers
- Summarize insights that could improve the loan evaluation process

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook

> Note: EDA shows *associations*, not causes. Findings should be validated, and real lending decisions must also consider fairness, privacy and banking regulations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("train.csv")
df.head()

## 2. Data Understanding
Let us check the size, column types, missing values, duplicates and summary statistics.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

### Observations (Data Understanding)
- The dataset contains the number of rows and columns shown by `df.shape` above, with applicant details, financial details and the final `Loan_Status`.
- Some columns (typically `LoanAmount`, `Credit_History`, `Loan_Amount_Term`, `Gender`, `Dependents`, `Self_Employed`) contain missing values, as shown by `df.isnull().sum()`.
- In `df.describe()`, if the `max` of `ApplicantIncome` or `LoanAmount` is far larger than the `mean`/`75%`, it indicates outliers (a few very high values).

In [ ]:
# Auto-summary from the real data
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
missing = df.isnull().sum()
print("\nColumns with missing values:")
print(missing[missing > 0].sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Data Cleaning
**Strategy**
- Categorical columns (`Gender`, `Married`, `Dependents`, `Self_Employed`, `Credit_History`, `Loan_Amount_Term`): fill with the **mode** (most frequent value)
- `LoanAmount`: fill with the **median**, because it is skewed and has outliers
- Remove duplicate rows (if any)
- Create a new feature `TotalIncome = ApplicantIncome + CoapplicantIncome`

In [ ]:
df_clean = df.copy()

for col in ["Gender", "Married", "Dependents", "Self_Employed", "Credit_History", "Loan_Amount_Term"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

df_clean["LoanAmount"] = df_clean["LoanAmount"].fillna(df_clean["LoanAmount"].median())

df_clean = df_clean.drop_duplicates()
df_clean["TotalIncome"] = df_clean["ApplicantIncome"] + df_clean["CoapplicantIncome"]

print("Missing values left:", df_clean.isnull().sum().sum())
print("Shape after cleaning:", df_clean.shape)

### Outlier check
Box plots are used to see extreme values in income and loan amount.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=df_clean["ApplicantIncome"], ax=ax[0]).set_title("ApplicantIncome")
sns.boxplot(y=df_clean["CoapplicantIncome"], ax=ax[1]).set_title("CoapplicantIncome")
sns.boxplot(y=df_clean["LoanAmount"], ax=ax[2]).set_title("LoanAmount")
plt.tight_layout()
plt.show()

Q1, Q3 = df_clean["ApplicantIncome"].quantile([0.25, 0.75])
IQR = Q3 - Q1
n_out = (df_clean["ApplicantIncome"] > Q3 + 1.5 * IQR).sum()
print(f"ApplicantIncome outliers (IQR rule): {n_out}")

**Observation:** The box plots show many points above the upper whisker for income and loan amount, which means a small number of applicants have very high values. These outliers are **kept** in the analysis because they can be genuine high earners; to reduce their effect on charts, the income plots below are zoomed to a normal range or shown on a log scale.

## 4. Exploratory Data Analysis

### 4.1 Loan Status Distribution

In [ ]:
sns.countplot(x="Loan_Status", data=df_clean, palette="Set2")
plt.title("Loan Status Distribution (Y = Approved, N = Rejected)")
plt.show()

rate = df_clean["Loan_Status"].value_counts(normalize=True) * 100
print(f"Approved: {rate['Y']:.1f}%  |  Rejected: {rate['N']:.1f}%")

**Observation:** More applications are approved than rejected (see the percentages printed above), so the dataset is moderately imbalanced towards approval. This matters if a prediction model is built later.

### 4.2 Income Distribution

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df_clean["ApplicantIncome"], kde=True, ax=ax[0]).set_title("ApplicantIncome")
sns.histplot(df_clean["CoapplicantIncome"], kde=True, ax=ax[1]).set_title("CoapplicantIncome")
sns.histplot(np.log1p(df_clean["TotalIncome"]), kde=True, ax=ax[2]).set_title("log(TotalIncome)")
plt.tight_layout()
plt.show()

print(df_clean[["ApplicantIncome", "CoapplicantIncome", "TotalIncome"]].describe().round(0))

**Observation:** Applicant and co-applicant incomes are right-skewed: most applicants earn in a lower range, while a few earn much more. Many co-applicants have zero income (no co-applicant). After a log transform, total income looks closer to a normal shape.

### 4.3 Credit History vs Loan Status

In [ ]:
sns.countplot(x="Credit_History", hue="Loan_Status", data=df_clean, palette="Set2")
plt.title("Credit History vs Loan Status")
plt.show()

ch = df_clean.groupby("Credit_History")["Loan_Status"].apply(lambda s: (s == "Y").mean() * 100)
print("Approval rate (%) by Credit_History:")
print(ch.round(1))

**Observation:** Credit history shows the clearest pattern. Applicants with `Credit_History = 1` have a much higher approval rate than those with `0` (see the printed rates). Credit history is strongly associated with approval in this data.

### 4.4 Property Area vs Loan Status

In [ ]:
sns.countplot(x="Property_Area", hue="Loan_Status", data=df_clean, palette="Set2")
plt.title("Property Area vs Loan Status")
plt.show()

pa = df_clean.groupby("Property_Area")["Loan_Status"].apply(lambda s: (s == "Y").mean() * 100)
print("Approval rate (%) by Property_Area:")
print(pa.round(1).sort_values(ascending=False))

**Observation:** Approval rates differ somewhat across Urban, Semiurban and Rural areas (see printed rates). The area with the highest rate is listed first in the output above. The difference is smaller than the credit history effect.

### 4.5 Income vs Loan Status

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x="Loan_Status", y="ApplicantIncome", data=df_clean, palette="Set2", ax=ax[0])
ax[0].set_ylim(0, 20000)
ax[0].set_title("ApplicantIncome by Loan Status (zoomed to 20000)")
sns.boxplot(x="Loan_Status", y="CoapplicantIncome", data=df_clean, palette="Set2", ax=ax[1])
ax[1].set_ylim(0, 10000)
ax[1].set_title("CoapplicantIncome by Loan Status (zoomed to 10000)")
plt.tight_layout()
plt.show()

print(df_clean.groupby("Loan_Status")[["ApplicantIncome", "CoapplicantIncome", "TotalIncome"]].median().round(0))

**Observation:** The income ranges of approved and rejected applicants overlap heavily (compare the medians printed above). Income alone does not appear to separate the two groups strongly.

### 4.6 Loan Amount vs Loan Status

In [ ]:
sns.boxplot(x="Loan_Status", y="LoanAmount", data=df_clean, palette="Set2")
plt.title("LoanAmount by Loan Status")
plt.show()

print(df_clean.groupby("Loan_Status")["LoanAmount"].median())

**Observation:** The loan amount distributions of approved and rejected applications are similar (compare the medians). Loan amount by itself shows only a weak association with the outcome.

### 4.7 Other Customer Categories (Education, Married, Self Employed, Gender, Dependents)

In [ ]:
cols = ["Education", "Married", "Self_Employed", "Gender", "Dependents"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, cols):
    sns.countplot(x=col, hue="Loan_Status", data=df_clean, palette="Set2", ax=ax)
    ax.set_title(f"{col} vs Loan Status")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

for col in cols:
    print(f"\nApproval rate (%) by {col}:")
    print((df_clean.groupby(col)["Loan_Status"].apply(lambda s: (s == "Y").mean() * 100)).round(1))

**Observation:** Approval rates vary slightly across these groups (see printed rates), for example by education level and marital status. These differences are modest compared with credit history. Variables like gender should be treated carefully: a difference in data is not a justification to use them in lending decisions (fairness and legal considerations).

### 4.8 Correlation Heatmap

In [ ]:
df_corr = df_clean.copy()
df_corr["Loan_Status_Num"] = (df_corr["Loan_Status"] == "Y").astype(int)
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term",
            "Credit_History", "TotalIncome", "Loan_Status_Num"]

plt.figure(figsize=(9, 6))
sns.heatmap(df_corr[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

print(df_corr[num_cols].corr()["Loan_Status_Num"].sort_values(ascending=False).round(3))

**Observation:** `Credit_History` has the strongest correlation with `Loan_Status_Num`, while income and loan amount have values close to zero. `ApplicantIncome` and `LoanAmount` are positively correlated with each other (higher income applicants tend to request larger loans). Correlation does not imply causation.

## 5. Key Insights
1. Most applications in the dataset are approved (see the approval percentage in section 4.1).
2. **Credit history is the strongest factor** associated with approval.
3. Income and loan amount, taken alone, show only weak association with approval.
4. Property area shows some variation in approval rates, with a smaller effect than credit history.
5. Income and loan amount are right-skewed with outliers, so medians and log transforms are more reliable than means.
6. Missing values were present in several columns and were handled using mode/median imputation.

## 6. Conclusion
The analysis suggests that a clean credit history is the most visible factor linked to loan approval, while income, loan amount and customer category show weaker patterns. Banks could pay special attention to credit history quality when evaluating applications, while checking other factors together rather than individually.

**Limitations:** EDA shows associations, not causes. The dataset is small, and lending decisions must also consider fairness, privacy and regulations.

**Next steps:** Build a classification model (e.g. Logistic Regression, Random Forest) to predict loan approval, and test it with proper train/test splits.